# Saved-analysis reproduction

We execute this results-only notebook in a new kernel in a separately installed environment. We read copied evidence, recalculate Step 35 and keep the original experiment unchanged. This notebook does not repeat the full project notebook, training or model generation.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import site
import sys

project = Path.cwd().resolve()
settings = json.loads((project / "replay_settings.json").read_text(encoding="utf-8"))
assert settings["mode"] == "results_reproduction"
run = Path(settings["run_folder"])
sys.path.insert(0, str(project / "analysis_code"))
for name, expected in settings["software_sha256"].items():
    assert hashlib.sha256((project / "analysis_code" / (name + ".py")).read_bytes()).hexdigest() == expected
kernel_info = {"pid": os.getpid(), "sys_executable": sys.executable, "sys_prefix": sys.prefix,
    "sys_base_prefix": sys.base_prefix, "python_version": platform.python_version(),
    "user_site_enabled": bool(site.ENABLE_USER_SITE),
    "packages": {dist.metadata["Name"].lower().replace("_", "-"): dist.version
                 for dist in importlib.metadata.distributions() if dist.metadata.get("Name")}}
(run / "kernel_info.json").write_text(json.dumps(kernel_info, sort_keys=True, indent=2) + "\n", encoding="utf-8")
network_attempts = []
def offline_analysis_guard(event, args):
    if event == "socket.connect":
        network_attempts.append({"event": event, "address": str(args[-1])})
        raise RuntimeError("Network access is disabled during saved-analysis reproduction.")
sys.addaudithook(offline_analysis_guard)
print("Results-only configuration loaded in the separate kernel.")


Results-only configuration loaded in the separate kernel.


In [2]:
for relative, expected in settings["stage_files"].items():
    assert hashlib.sha256((project / relative).read_bytes()).hexdigest() == expected, relative
print("Saved inputs match the recorded versions.")


Saved inputs match the recorded versions.


In [3]:
from step35_analysis_runner import analyze
configuration = json.loads((project / "analysis_configuration.json").read_text(encoding="utf-8"))
result = analyze(project, configuration)
assert result["new_model_calls"] == result["new_predictions"] == result["new_reference_queries"] == 0
(run / "replay_result.json").write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("Saved analysis recalculated.")


Saved analysis recalculated.


In [4]:
integrity = []
for relative, expected in settings["stage_files"].items():
    actual = hashlib.sha256((project / relative).read_bytes()).hexdigest()
    integrity.append({"path": relative, "sha256_before": expected, "sha256_after": actual, "unchanged": actual == expected})
assert all(item["unchanged"] for item in integrity)
(run / "staged_input_integrity.json").write_text(json.dumps(integrity, sort_keys=True, indent=2) + "\n", encoding="utf-8")
(run / "network_attempts.json").write_text(json.dumps(network_attempts, sort_keys=True, indent=2) + "\n", encoding="utf-8")
assert not network_attempts
print("Copied inputs are unchanged. No network requests were made by the analysis.")


Copied inputs are unchanged. No network requests were made by the analysis.
